# 🌀 Pista A — El huracán: de 50 futuros posibles a una alerta
**Código que Salva Vidas · DevFest Santo Domingo 2026 · Taller · Nivel principiante**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alcastaro/codigo-que-salva-vidas/blob/main/taller/pista_A_huracan.ipynb)

El 28 de octubre de 2025, **Melissa** tocó tierra en Jamaica como huracán categoría 5, con vientos de 160 nudos (unos 300 km/h). Fue el más fuerte que ha golpeado la isla.
Según Google DeepMind, su modelo **WeatherNext** anticipó esa llegada como categoría 5 **cinco días antes, con un 80 % de confianza**.

En esta pista haces tú lo que hicieron los pronosticadores:

| Paso | Qué haces | Producto de Google |
|---|---|---|
| 1 | Ver los **50 escenarios** que el modelo de IA dibujó tres días antes | Weather Lab (DeepMind) |
| 2 | Calcular la **probabilidad** de que el ojo pase cerca de un lugar | Weather Lab |
| 3 | Ver qué **ríos** marcaban peligro de inundación | Flood Hub |
| 4 | Convertir todo en una **alerta** que se entienda en 3 segundos | Gemini (AI Studio) |

**No necesitas saber programar.** Ejecuta cada celda con ▶️ (o `Shift + Enter`) en orden y lee el texto de arriba.

In [ ]:
# Instala lo que falta en Colab y trae las funciones de apoyo del taller (~30 s)
!pip -q install rasterio pystac-client planetary-computer google-genai
!wget -q -N https://raw.githubusercontent.com/alcastaro/codigo-que-salva-vidas/main/taller/ayuda.py
import ayuda, numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
print("Listo ✅")

## Paso 1 · 50 futuros posibles

Un pronóstico moderno no dibuja **una** trayectoria: dibuja **muchas**. Cada una es un futuro posible, y a ese conjunto se le llama **ensamble**.
Si todas las líneas van juntas, el modelo está seguro. Si se abren como un abanico, hay incertidumbre.

Vamos a cargar el pronóstico que el modelo experimental de ciclones de DeepMind emitió el **25 de octubre de 2025 a las 00:00 UTC**, tres días antes de la llegada a tierra.
Lo comparamos con la **trayectoria real**, publicada después por el Centro Nacional de Huracanes de EE. UU. (NHC).

In [ ]:
# @title Elige la fecha en que se emitió el pronóstico { run: "auto" }
emision = "2025_10_25T00"  # @param ["2025_10_22T00", "2025_10_25T00", "2025_10_27T00"]

ens = ayuda.leer_weatherlab(f"data/melissa/weatherlab_FNV3P2_ensemble_{emision}.csv")
real = ayuda.leer_trayectoria_real()
print(f"{ens['sample'].nunique()} escenarios · {len(ens)} puntos (uno cada 6 horas por escenario)")
ens.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ayuda.mapa_base(ax)
for _, t in ens.groupby("sample"):
    ax.plot(t.lon, t.lat, color="#4285f4", alpha=0.25, lw=1, zorder=2)
ax.plot(real.lon, real.lat, color="#ea4335", lw=3, zorder=3, label="Trayectoria real (NHC)")
ax.plot([], [], color="#4285f4", label="Los 50 escenarios del modelo")
llegada = real.loc[real.viento_kt.idxmax()]
ax.scatter(llegada.lon, llegada.lat, s=120, color="#f9ab00", ec="#1e1e1e", zorder=4,
           label=f"Máximo: {llegada.viento_kt} nudos ({ayuda.categoria(llegada.viento_kt)})")
ax.set_title(f"Melissa · pronóstico emitido el {emision[:10].replace('_', '-')}")
ax.legend(loc="lower left"); plt.show()

**🧠 Para pensar:** ¿el abanico azul contiene la línea roja? Cambia la fecha de emisión arriba (22, 25 o 27 de octubre) y vuelve a ejecutar.
¿Cómo cambia el abanico a medida que se acerca la tormenta?

## Paso 2 · ¿Qué tan probable es que me toque?

Con 50 escenarios, la probabilidad se calcula **contando**: si 40 de 50 pasan a menos de 100 km de un lugar, la probabilidad es del 80 %.
Elige un lugar en el desplegable o escribe tus propias coordenadas.

In [ ]:
# @title Elige un lugar { run: "auto" }
lugar = "Black River, Jamaica"  # @param ["Black River, Jamaica", "Kingston, Jamaica", "Santo Domingo, RD", "Santiago, RD", "Puerto Príncipe, Haití"]
radio_km = 100  # @param {type:"slider", min:25, max:300, step:25}

lat, lon = ayuda.LUGARES[lugar]
p = ayuda.probabilidad_paso(ens, lat, lon, radio_km)
print(f"📍 {lugar}")
print(f"   {p['pasan_cerca']} de {p['escenarios']} escenarios pasan a menos de {radio_km} km → probabilidad {p['probabilidad']:.0%}")
if p["pasan_cerca"]:
    print(f"   Viento máximo típico al pasar: {p['viento_mediano_kt']:.0f} nudos ({ayuda.categoria(p['viento_mediano_kt'])})")
    print(f"   Probabilidad de que llegue como categoría 5: {p['prob_cat5']:.0%}")
    print(f"   Momento más probable: {p['llegada_mas_probable']:%d-%b %H:%M} UTC")

## Paso 3 · ¿Qué ríos se van a desbordar? (Flood Hub)

**Flood Hub** pronostica crecidas de ríos hasta 7 días antes en más de 150 países, incluso en ríos **sin estaciones de medición**.
Lo logra con un modelo de IA (dos redes neuronales LSTM en cadena) entrenado con datos de miles de ríos de todo el mundo.

La API de Flood Hub requiere una key aprobada, así que el facilitador del taller extrajo una **copia** de lo que marcaba durante Melissa.
Para verlo en vivo, abre [sites.research.google/floods](https://sites.research.google/floods).

In [ ]:
try:
    fh = ayuda.leer_floodhub()
    est = fh["estado"]
    print("Extraído:", fh["meta"]["extraido"], "· estado a", fh["meta"]["melissa_cutoff"])
    print(f"{len(fh['medidores'])} puntos de pronóstico en {', '.join(fh['meta']['regiones'])}")
    display(est.filter(regex="gaugeId|severity|forecastTrend|issuedTime").head(15))
except Exception as e:
    print("La copia de Flood Hub aún no está disponible en este repositorio:", e)
    fh = None

In [ ]:
if fh is not None and len(fh["estado"]):
    m = fh["medidores"].merge(fh["estado"], left_on="gaugeId", right_on="gaugeId", how="left")
    col = {"EXTREME": "#ea4335", "SEVERE": "#f9ab00", "ABOVE_NORMAL": "#4285f4", "NO_FLOODING": "#34a853"}
    fig, ax = plt.subplots(figsize=(9, 6)); ayuda.mapa_base(ax)
    for sev, g in m.groupby(m.get("severity", pd.Series(["?"] * len(m))).fillna("SIN_DATO")):
        ax.scatter(g["location.longitude"], g["location.latitude"], s=25, color=col.get(sev, "#9e9e9e"),
                   ec="#1e1e1e", lw=0.4, zorder=3, label=sev)
    ax.plot(real.lon, real.lat, color="#1e1e1e", lw=1.5, ls="--", zorder=2, label="Melissa")
    ax.legend(loc="lower left", fontsize=8); ax.set_title("Flood Hub durante Melissa"); plt.show()

### 🔑 Tu API key de Gemini (2 minutos, solo la primera vez)
1. Abre **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** y pulsa **Create API key**. Es gratis.
2. En Colab, abre el panel de la **llave 🔑 (Secretos)** a la izquierda.
3. Pulsa **Add new secret**: nombre `GOOGLE_API_KEY`, valor tu key, y activa **Notebook access**.

> Atajo: en ese mismo panel, **Gemini API keys → Import key from Google AI Studio** lo hace por ti.
>
> **Nunca pegues una key dentro de una celda.** Si compartes el notebook o lo subes a GitHub, la key viaja con él.

## Paso 4 · De los números a una alerta

Un pronóstico excelente no salva a nadie si la persona no lo entiende. Le pasamos a **Gemini** los números de los pasos anteriores, y él redacta la alerta siguiendo reglas claras.
La alerta lleva además tres campos del **Protocolo Común de Alertas (CAP)**: severidad, urgencia y certeza. CAP es el estándar con el que las autoridades publican alertas, y es el formato que Google usa para mostrarlas en Búsqueda y Maps.

In [ ]:
cliente = ayuda.cliente_gemini()
modelo = ayuda.elegir_modelo(cliente, "flash")
print("Modelo:", modelo)

datos = {
    "evento": "Huracán Melissa",
    "pronostico_emitido": emision,
    "lugar": lugar,
    "probabilidad_paso_a_menos_de_km": {"km": radio_km, "probabilidad": round(p["probabilidad"], 2)},
    "viento_esperado": ayuda.categoria(p["viento_mediano_kt"] or 0),
    "probabilidad_categoria_5": round(p["prob_cat5"], 2),
    "llegada_mas_probable_utc": p["llegada_mas_probable"],
    "rios_en_alerta_flood_hub": int(fh["estado"].get("severity", pd.Series()).isin(["EXTREME", "SEVERE"]).sum()) if fh else "sin datos",
}
alerta = ayuda.redactar_alerta(cliente, modelo, datos)
ayuda.mostrar_alerta(alerta)

### 🧪 Extra: la misma alerta con Gemma, un modelo abierto

**Gemma** es la familia de modelos abiertos de Google. Sus versiones pequeñas corren **sin internet**, en un teléfono o una Raspberry Pi.
Así lo hizo *Acuífero·Vigía*, ganador del premio Global Resilience del Gemma 4 Good Challenge: un sistema de alerta de inundaciones que funciona sin conexión.
¿La alerta de Gemma es tan buena como la de Gemini?

In [ ]:
modelo_gemma = ayuda.elegir_modelo(cliente, "gemma")
print("Modelo:", modelo_gemma)
ayuda.mostrar_alerta(ayuda.redactar_alerta(cliente, modelo_gemma, datos))

## 🏁 Retos (si te sobra tiempo)
1. Emite la alerta para **Santo Domingo** con el pronóstico del **22 de octubre**. ¿Qué certeza le pone Gemini? ¿Tiene sentido?
2. Edita `ayuda.INSTRUCCION_ALERTA` para que la alerta salga en **creole haitiano** o en **inglés jamaicano**, y vuelve a ejecutar el paso 4.
3. ¿Cuántos escenarios predijeron categoría 5 en Jamaica el 22 de octubre? ¿Y el 27?

---
**Fuentes y licencias.** Pronósticos: Google DeepMind Weather Lab, modelo experimental de ciclones, datos históricos con licencia CC BY 4.0. Es un pronóstico de investigación, no un producto oficial.
Trayectoria real: NOAA / National Hurricane Center (best track AL132025). Inundaciones: Google Flood Forecasting API (uso no comercial).
Contornos: Natural Earth. **Para alertas reales, consulta siempre a tu servicio meteorológico nacional.**